<a href="https://colab.research.google.com/github/Aaryant74/Data_Science_Assignments/blob/main/19_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Task 1 : Data Cleaning: Remove duplicates, handle missing reviews if any, preprocess text (lowercasing, stopwords removal).**


In [ ]:
import pandas as pd

# Load TSV file
df = pd.read_csv('/content/amazonreviews.tsv', sep='\t')

# Check data
df.head()

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates()

print("Duplicates removed. New shape:", df.shape)

In [ ]:
# Handle Missing Values
# Check missing values
print(df.isnull().sum())

# Drop rows where review text is missing
df = df.dropna(subset=['review'])

In [ ]:
# Text Processing
# Importing Libraries that are required
import nltk
from nltk.corpus import stopwords
import re

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

In [ ]:
# Cleaning the text
def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove special characters & numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # Remove stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

# Apply cleaning
df['cleaned_review'] = df['review'].apply(clean_text)

df[['review', 'cleaned_review']].head()

**Task 2 : Exploratory Analysis: Word clouds, sentiment distribution, most common positive/negative words.**

In [ ]:
# For Exploratory Analysis
!pip install wordcloud

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from collections import Counter

In [ ]:
# Word Cloud
text = " ".join(df['cleaned_review'])

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)

plt.figure(figsize=(10,5))
plt.imshow(wordcloud)
plt.axis('off')
plt.title("Word Cloud")
plt.show()

In [ ]:
# Sentiment Analysis
# Using Testblob
!pip install textblob
from textblob import TextBlob

In [ ]:
# Creating Sentiment Function
def get_sentiment(text):
    score = TextBlob(text).sentiment.polarity
    if score > 0:
        return 'Positive'
    elif score < 0:
        return 'Negative'
    else:
        return 'Neutral'

In [ ]:
# Now applying Function
df['sentiment'] = df['cleaned_review'].apply(get_sentiment)
df['sentiment'].value_counts()

In [ ]:
# Sentiment Distribution (Graph)
df['sentiment'].value_counts().plot(kind='bar')
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

In [ ]:
# Most Common Positive Words
positive_words = " ".join(df[df['sentiment']=='Positive']['cleaned_review']).split()

common_pos = Counter(positive_words).most_common(10)
print("Top Positive Words:", common_pos)

In [ ]:
# Most Common Negative Words
negative_words = " ".join(df[df['sentiment']=='Negative']['cleaned_review']).split()

common_neg = Counter(negative_words).most_common(10)
print("Top Negative Words:", common_neg)

In Above task, i have acheived :

1. Visualized frequent words (Word Cloud)

2. Classified reviews into Positive/Negative/Neutral

3. Analyzed distribution of sentiments

4. Extracted most common words per
sentiment

**Task 3 : Model Development: Use NLP techniques (TF-IDF, Word2Vec, or BERT embeddings) with models like Logistic Regression, SVM, or Neural Networks.**

In [ ]:
# Preparing Data
from sklearn.model_selection import train_test_split

# X = input text (cleaned reviews)
X = df['cleaned_review']

# y = output labels (Positive / Negative / Neutral)
y = df['sentiment']

# Split data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Converting Text -- TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert text data into numerical form using TF-IDF
# max_features=5000 → use top 5000 important words
tfidf = TfidfVectorizer(max_features=5000)

# Learn vocabulary from training data and transform it
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform test data using same vocabulary
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
# Logistic Regression Model
from sklearn.linear_model import LogisticRegression

# Create Logistic Regression model
lr_model = LogisticRegression()

# Train model on TF-IDF features
lr_model.fit(X_train_tfidf, y_train)

# Predict sentiments on test data
y_pred_lr = lr_model.predict(X_test_tfidf)

In [ ]:
# SVM Model
from sklearn.svm import SVC

# Create Support Vector Machine model
svm_model = SVC()

# Train the SVM model
svm_model.fit(X_train_tfidf, y_train)

# Predict using SVM
y_pred_svm = svm_model.predict(X_test_tfidf)

In [ ]:
# Evaluation
from sklearn.metrics import accuracy_score, classification_report

# Logistic Regression Performance
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
# Shows precision, recall, F1-score for each class
print(classification_report(y_test, y_pred_lr))

# SVM Performance
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

In [ ]:
# Neural Networks
from sklearn.neural_network import MLPClassifier

# Create simple Neural Network with one hidden layer
nn_model = MLPClassifier(hidden_layer_sizes=(100,))

# Train model
nn_model.fit(X_train_tfidf, y_train)

# Predict on test data
y_pred_nn = nn_model.predict(X_test_tfidf)

# Check accuracy
print("Neural Network Accuracy:", accuracy_score(y_test, y_pred_nn))

**Task 4 : Validation: Use train/test split, cross-validation, and metrics like accuracy, F1-score.**

In [ ]:
# Train/Test Split
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
# 80% data for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Cross Validation
from sklearn.model_selection import cross_val_score

# Perform 5-fold cross-validation on Logistic Regression
# Data is split into 5 parts, model is trained 5 times
cv_scores = cross_val_score(lr_model, X_train_tfidf, y_train, cv=5)

# Print accuracy for each fold
print("Cross-validation scores:", cv_scores)

# Average accuracy across all folds
print("Mean CV Accuracy:", cv_scores.mean())

In [ ]:
# Evaluation Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Accuracy → overall correct predictions
accuracy = accuracy_score(y_test, y_pred_lr)

# Precision → correct positive predictions
precision = precision_score(y_test, y_pred_lr, average='weighted')

# Recall → how many actual positives were found
recall = recall_score(y_test, y_pred_lr, average='weighted')

# F1-score → balance between precision and recall
f1 = f1_score(y_test, y_pred_lr, average='weighted')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

In [ ]:
# Detailed Classification report
from sklearn.metrics import classification_report

# Shows precision, recall, F1-score for each class
print(classification_report(y_test, y_pred_lr))

***“Cross-validation provides a more robust estimate of model performance compared to a single train-test split.”***